In [1]:
import pulp
import pandas as pd
from datetime import datetime

# --- 1. CONFIGURATION: YOUR AGENT LIST --- # Check these workdays! 0=Mon, 4=Fri, 5=Sat, 6=Sun.
agents_db = {}
for i in range(1, 51):
    if i <= 20: # 1-20: Morning (Mon-Fri)
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (0, 9)}
    elif i <= 35: # 21-35: Afternoon (Mon-Fri)
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (9, 23)}
    else: # 36-50: Everyone else (Full Week)
        agents_db[f"Agent_{i:02d}"] = {"type": "PT", "workdays": [0,1,2,3,4,5,6], "window": (0, 23)}

def solve_day(date_val, demand_dict):
    weekday = date_val.weekday()
    avail = [a for a, info in agents_db.items() if weekday in info['workdays']]
    
    # DEBUG: See how many agents are available
    print(f"--- Day: {date_val.date()} (Weekday {weekday}) | Agents Available: {len(avail)} ---")
    
    if len(avail) == 0:
        print("!!! WARNING: No agents available for this day of the week.")
        return []

    prob = pulp.LpProblem(f"Sch_{date_val.date()}", pulp.LpMinimize)
    slack_vars = {}
    choices = {}
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        for s in range(s_min, s_max + 1):
            choices[(a, s)] = pulp.LpVariable(f"S_{a}_{s}_{date_val.date()}", cat="Binary")

    # Constraint: Max 1 shift
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        prob += pulp.lpSum([choices[(a, s)] for s in range(s_min, s_max + 1)]) <= 1

    # Constraint: Coverage with "Slack" (allows it to finish even if understaffed)
    for h in range(24):
        coverage = []
        for a in avail:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            s_min, s_max = agents_db[a]['window']
            for s in range(s_min, s_max + 1):
                if s <= h < s + dur:
                    if agents_db[a]['type'] == "FT" and h == (s + 4): # Lunch
                        coverage.append(choices[(a, s)] * 0)
                    else: coverage.append(choices[(a, s)] * 1)
        
        # This 'slack' variable makes the code 'Optimal' even if you don't have enough staff
        slack = pulp.LpVariable(f"Slack_{h}", lowBound=0)
        slack_vars[h]=slack
        prob += pulp.lpSum(coverage) + slack >= demand_dict.get(h, 0)

   # Goal: Minimize agents used AND heavily penalize understaffing (Slack)
   # We multiply slack by 100 to make it "expensive" for the computer to ignore demand
    prob += pulp.lpSum([choices[v] for v in choices]) + pulp.lpSum([slack_vars[h] * 100 for h in range(24)])
    prob.solve(pulp.PULP_CBC_CMD(msg=1,timeLimit=15))
    
    res = []
    print(f"Status: {pulp.LpStatus[prob.status]}") # Should say 'Optimal'
    
    for (a, s) in choices:
        if pulp.value(choices[(a, s)]) == 1:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            res.append({"Date": date_val.date(), "Agent": a, "Start": f"{s:02d}:00", "End": f"{(s+dur)%24:02d}:00"})
    
    print(f"Assignments made: {len(res)}")
    return res

# --- UPDATED LOADING AND RUNNING SECTION ---
try:
    # 1. Load the file
    df = pd.read_excel("need.xlsx")
    
    # 2. Let's see what the computer actually sees (Diagnostic)
    actual_col_count = len(df.columns)
    print(f"I found {actual_col_count} columns in your Excel file.")
    print(f"Column names found: {list(df.columns)}")

       # 3. Create a cleaned version of the data
    clean_df = pd.DataFrame()
   
   # Column A: Date
    clean_df['Date'] = pd.to_datetime(df.iloc[:, 0]) 
   
   # Column C: Hour (Converting "09:00" string to integer 9)
   # We convert to string, split at the colon, take the first part, and make it a number
    clean_df['Hour'] = pd.to_numeric(df.iloc[:, 2].astype(str).str.split(':').str[0], errors='coerce')
   
   # Column D: FTE_Brut (Ensuring it is a number, not text)
    clean_df['FTE_Brut'] = pd.to_numeric(df.iloc[:, 4], errors='coerce').fillna(0)

   # Clean up any rows that aren't valid numbers
    clean_df = clean_df.dropna(subset=['Hour'])


    
    print(f"Successfully processed {len(clean_df)} rows of data.")

    
    all_res = []
    for d in clean_df['Date'].unique():
        # Convert numpy datetime back to pandas timestamp for the solver
        ts = pd.to_datetime(d)
        day_data = clean_df[clean_df['Date'] == d]
        
        # Map Hour to FTE
        demand_dict = dict(zip(day_data['Hour'], day_data['FTE_Brut']))
        
        
        #---DEBUG PRINT----
        print(f"Total Demand for {ts.date()}:{sum(demand_dict.values())}")
        print(f"Hours with demand:{list(demand_dict.keys())}")
    
        # Solve for the day
        all_res.extend(solve_day(ts, demand_dict))

    print (demand_dict)
        
    if len(all_res) > 0:
        output_df = pd.DataFrame(all_res)
        output_df.to_excel("final_schedulev2.xlsx", index=False)
        print("\n*** SUCCESS! Check final_schedulev2.xlsx ***")
    else:
        print("\n*** ERROR: No assignments were generated. Check the text above for warnings. ***")

except Exception as e:
    # This will tell us EXACTLY what went wrong
    import traceback
    print("\n*** CRITICAL ERROR DETAILS ***")
    print(traceback.format_exc())

I found 15 columns in your Excel file.
Column names found: ['Date', 'Day', 'Hour', 'Volume', 'Sizing FTEs', 'Calls/Day/FTE', 'Cases/Day/FTE', 'Service Level %', 'Avg Wait Time (s)', 'Occupancy %', 'P(Wait) %', 'Immediate Answer %', 'Shift', 'Weekend', 'Unnamed: 14']
Successfully processed 2208 rows of data.
Total Demand for 2026-05-01:164.12307692307695
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-05-01 (Weekday 4) | Agents Available: 50 ---
Status: Optimal
Assignments made: 31
Total Demand for 2026-05-02:122.21538461538462
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-05-02 (Weekday 5) | Agents Available: 15 ---
Status: Optimal
Assignments made: 15
Total Demand for 2026-05-03:123.81538461538462
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-05-03 (Weekday 6) | Agents Av

Status: Optimal
Assignments made: 28
Total Demand for 2026-06-02:145.90769230769232
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-06-02 (Weekday 1) | Agents Available: 50 ---
Status: Optimal
Assignments made: 28
Total Demand for 2026-06-03:157.53846153846158
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-06-03 (Weekday 2) | Agents Available: 50 ---
Status: Optimal
Assignments made: 30
Total Demand for 2026-06-04:154.2769230769231
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-06-04 (Weekday 3) | Agents Available: 50 ---
Status: Optimal
Assignments made: 29
Total Demand for 2026-06-05:146.70769230769233
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-06-05 (Weekday 4) | Agents Available: 50 ---
Status: O

Status: Optimal
Assignments made: 15
Total Demand for 2026-07-06:179.6307692307692
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-07-06 (Weekday 0) | Agents Available: 50 ---
Status: Optimal
Assignments made: 33
Total Demand for 2026-07-07:174.7692307692308
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-07-07 (Weekday 1) | Agents Available: 50 ---
Status: Optimal
Assignments made: 32
Total Demand for 2026-07-08:190.46153846153842
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-07-08 (Weekday 2) | Agents Available: 50 ---
Status: Optimal
Assignments made: 35
Total Demand for 2026-07-09:187.13846153846148
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-07-09 (Weekday 3) | Agents Available: 50 ---
Status: Op

In [2]:
import pulp
import pandas as pd
from datetime import datetime

# --- 1. CONFIGURATION: YOUR AGENT LIST --- # Check these workdays! 0=Mon, 4=Fri, 5=Sat, 6=Sun.
agents_db = {}
for i in range(1, 51):
    if i <= 20: # 1-20: Morning (Mon-Fri)
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (0, 9)}
    elif i <= 35: # 21-35: Afternoon (Mon-Fri)
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (9, 23)}
    else: # 36-50: Everyone else (Full Week)
        agents_db[f"Agent_{i:02d}"] = {"type": "PT", "workdays": [0,1,2,3,4,5,6], "window": (0, 23)}

def solve_day(date_val, demand_dict):
    weekday = date_val.weekday()
    avail = [a for a, info in agents_db.items() if weekday in info['workdays']]
    
    # DEBUG: See how many agents are available
    print(f"--- Day: {date_val.date()} (Weekday {weekday}) | Agents Available: {len(avail)} ---")
    
    if len(avail) == 0:
        print("!!! WARNING: No agents available for this day of the week.")
        return []

    prob = pulp.LpProblem(f"Sch_{date_val.date()}", pulp.LpMinimize)
    slack_vars = {}
    choices = {}
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        for s in range(s_min, s_max + 1):
            choices[(a, s)] = pulp.LpVariable(f"S_{a}_{s}_{date_val.date()}", cat="Binary")

    # Constraint: Max 1 shift
    for a in avail:
        s_min, s_max = agents_db[a]['window']
        prob += pulp.lpSum([choices[(a, s)] for s in range(s_min, s_max + 1)]) <= 1

    # Constraint: Coverage with "Slack" (allows it to finish even if understaffed)
    for h in range(24):
        coverage = []
        for a in avail:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            s_min, s_max = agents_db[a]['window']
            for s in range(s_min, s_max + 1):
                if s <= h < s + dur:
                    if agents_db[a]['type'] == "FT" and h == (s + 4): # Lunch
                        coverage.append(choices[(a, s)] * 0)
                    else: coverage.append(choices[(a, s)] * 1)
        
        # This 'slack' variable makes the code 'Optimal' even if you don't have enough staff
        slack = pulp.LpVariable(f"Slack_{h}", lowBound=0)
        slack_vars[h]=slack
        prob += pulp.lpSum(coverage) + slack >= demand_dict.get(h, 0)

   # Goal: Minimize agents used AND heavily penalize understaffing (Slack)
   # We multiply slack by 100 to make it "expensive" for the computer to ignore demand
    prob += pulp.lpSum([choices[v] for v in choices]) + pulp.lpSum([slack_vars[h] * 100 for h in range(24)])
    prob.solve(pulp.PULP_CBC_CMD(msg=1,timeLimit=15))
    
    res = []
    print(f"Status: {pulp.LpStatus[prob.status]}") # Should say 'Optimal'
    
    for (a, s) in choices:
        if pulp.value(choices[(a, s)]) == 1:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            res.append({"Date": date_val.date(), "Agent": a, "Start": f"{s:02d}:00", "End": f"{(s+dur)%24:02d}:00"})
    
    print(f"Assignments made: {len(res)}")
    return res

# --- UPDATED LOADING AND RUNNING SECTION ---
try:
    # 1. Load the file
    df = pd.read_excel("need.xlsx")
    
    # 2. Let's see what the computer actually sees (Diagnostic)
    actual_col_count = len(df.columns)
    print(f"I found {actual_col_count} columns in your Excel file.")
    print(f"Column names found: {list(df.columns)}")

       # 3. Create a cleaned version of the data
    clean_df = pd.DataFrame()
   
   # Column A: Date
    clean_df['Date'] = pd.to_datetime(df.iloc[:, 0]) 
   
   # Column C: Hour (Converting "09:00" string to integer 9)
   # We convert to string, split at the colon, take the first part, and make it a number
    clean_df['Hour'] = pd.to_numeric(df.iloc[:, 2].astype(str).str.split(':').str[0], errors='coerce')
   
   # Column D: FTE_Brut (Ensuring it is a number, not text)
    clean_df['FTE_Brut'] = pd.to_numeric(df.iloc[:, 4], errors='coerce').fillna(0)

   # Clean up any rows that aren't valid numbers
    clean_df = clean_df.dropna(subset=['Hour'])


    
    print(f"Successfully processed {len(clean_df)} rows of data.")

    
    all_res = []
    for d in clean_df['Date'].unique():
        # Convert numpy datetime back to pandas timestamp for the solver
        ts = pd.to_datetime(d)
        day_data = clean_df[clean_df['Date'] == d]
        
        # Map Hour to FTE
        demand_dict = dict(zip(day_data['Hour'], day_data['FTE_Brut']))
        
        
        #---DEBUG PRINT----
        print(f"Total Demand for {ts.date()}:{sum(demand_dict.values())}")
        print(f"Hours with demand:{list(demand_dict.keys())}")
    
        # Solve for the day
        all_res.extend(solve_day(ts, demand_dict))

    print (demand_dict)
        
    if len(all_res) > 0:
        output_df = pd.DataFrame(all_res)
        output_df.to_excel("final_schedulev2.xlsx", index=False)
        print("\n*** SUCCESS! Check final_schedulev2.xlsx ***")
    else:
        print("\n*** ERROR: No assignments were generated. Check the text above for warnings. ***")

except Exception as e:
    # This will tell us EXACTLY what went wrong
    import traceback
    print("\n*** CRITICAL ERROR DETAILS ***")
    print(traceback.format_exc())

I found 15 columns in your Excel file.
Column names found: ['Date', 'Day', 'Hour', 'Volume', 'Sizing FTEs', 'Calls/Day/FTE', 'Cases/Day/FTE', 'Service Level %', 'Avg Wait Time (s)', 'Occupancy %', 'P(Wait) %', 'Immediate Answer %', 'Shift', 'Weekend', 'Unnamed: 14']
Successfully processed 2208 rows of data.
Total Demand for 2026-05-01:164.12307692307695
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-05-01 (Weekday 4) | Agents Available: 50 ---
Status: Optimal
Assignments made: 31
Total Demand for 2026-05-02:122.21538461538462
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-05-02 (Weekday 5) | Agents Available: 15 ---
Status: Optimal
Assignments made: 15
Total Demand for 2026-05-03:123.81538461538462
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-05-03 (Weekday 6) | Agents Av

Status: Optimal
Assignments made: 28
Total Demand for 2026-06-02:145.90769230769232
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-06-02 (Weekday 1) | Agents Available: 50 ---
Status: Optimal
Assignments made: 28
Total Demand for 2026-06-03:157.53846153846158
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-06-03 (Weekday 2) | Agents Available: 50 ---
Status: Optimal
Assignments made: 30
Total Demand for 2026-06-04:154.2769230769231
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-06-04 (Weekday 3) | Agents Available: 50 ---
Status: Optimal
Assignments made: 29
Total Demand for 2026-06-05:146.70769230769233
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-06-05 (Weekday 4) | Agents Available: 50 ---
Status: O

Status: Optimal
Assignments made: 15
Total Demand for 2026-07-06:179.6307692307692
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-07-06 (Weekday 0) | Agents Available: 50 ---
Status: Optimal
Assignments made: 33
Total Demand for 2026-07-07:174.7692307692308
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-07-07 (Weekday 1) | Agents Available: 50 ---
Status: Optimal
Assignments made: 32
Total Demand for 2026-07-08:190.46153846153842
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-07-08 (Weekday 2) | Agents Available: 50 ---
Status: Optimal
Assignments made: 35
Total Demand for 2026-07-09:187.13846153846148
Hours with demand:[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
--- Day: 2026-07-09 (Weekday 3) | Agents Available: 50 ---
Status: Op

In [3]:
import pulp
import pandas as pd
from datetime import datetime

# --- 1. CONFIGURATION ---
agents_db = {}
for i in range(1, 51):
    if i <= 20: 
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (0, 9)}
    elif i <= 35: 
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (9, 23)}
    else: 
        agents_db[f"Agent_{i:02d}"] = {"type": "PT", "workdays": [0,1,6], "window": (0, 23)}

def solve_full_period(clean_df):
    all_dates = sorted(clean_df['Date'].unique())
    prob = pulp.LpProblem("Global_Schedule", pulp.LpMinimize)
    
    choices = {}      # (agent, date, start_hour)
    slack_vars = {}   # (date, hour)
    agent_daily_work = {} # (agent, date) -> Binary (1 if working that day)

    # --- 2. VARIABLE CREATION ---
    for d in all_dates:
        ts = pd.to_datetime(d)
        weekday = ts.weekday()
        avail_today = [a for a, info in agents_db.items() if weekday in info['workdays']]
        
        for a in avail_today:
            s_min, s_max = agents_db[a]['window']
            # Track if agent works at all on this day
            agent_daily_work[(a, d)] = pulp.LpVariable(f"Work_{a}_{d}", cat="Binary")
            
            day_choices = []
            for s in range(s_min, s_max + 1):
                v = pulp.LpVariable(f"S_{a}_{d}_{s}", cat="Binary")
                choices[(a, d, s)] = v
                day_choices.append(v)
            
            # Constraint: Max 1 shift per day
            prob += pulp.lpSum(day_choices) == agent_daily_work[(a, d)]

    # --- 3. COVERAGE & SLACK ---
    for d in all_dates:
        day_data = clean_df[clean_df['Date'] == d]
        demand_dict = dict(zip(day_data['Hour'], day_data['FTE_Brut']))
        
        for h in range(24):
            coverage = []
            for (a, date, s), var in choices.items():
                if date == d:
                    dur = 8 if agents_db[a]['type'] == "FT" else 4
                    if s <= h < s + dur:
                        # Handle Lunch for FT
                        if agents_db[a]['type'] == "FT" and h == (s + 4):
                            continue
                        coverage.append(var)
            
            slack = pulp.LpVariable(f"Slack_{d}_{h}", lowBound=0)
            slack_vars[(d, h)] = slack
            prob += pulp.lpSum(coverage) + slack >= demand_dict.get(h, 0)

    # --- 4. LAYER: EQUITY (Rebalancing) ---
    # Target: Total shifts needed / Number of agents
    total_days = len(all_dates)
    for a in agents_db:
        # Number of shifts assigned to this agent over the period
        agent_total_shifts = pulp.lpSum([agent_daily_work[(a, d)] for d in all_dates if (a, d) in agent_daily_work])
        
        # Helper variables to measure deviation from a "fair" load
        # We penalize agents working way more or way less than 4 shifts a week (as an example)
        over_work = pulp.LpVariable(f"Over_{a}", lowBound=0)
        prob += agent_total_shifts <= 5 + over_work # Penalty if more than 5 shifts/week

    # --- 5. LAYER: NO 3-DAY GAPS ---
    # For every agent, in any rolling 3-day window they are available, they must work >= 1 day
    for a in agents_db:
        workdays = agents_db[a]['workdays']
        for i in range(len(all_dates) - 2):
            window = all_dates[i:i+3]
            # Only apply if the agent is actually contracted to work these days
            valid_days_in_window = [d for d in window if pd.to_datetime(d).weekday() in workdays]
            
            if len(valid_days_in_window) == 3:
                prob += pulp.lpSum([agent_daily_work[(a, d)] for d in valid_days_in_window]) >= 1

    # --- 6. OBJECTIVE ---
    # 1. Minimize Understaffing (Slack) - Priority 1
    # 2. Minimize Total Agents used - Priority 2
    # 3. Minimize Overwork - Priority 3
    prob += (
        pulp.lpSum(slack_vars.values()) * 100 + 
        pulp.lpSum(choices.values()) * 1 +
        pulp.lpSum([pulp.LpVariable(f"dev_{a}") for a in agents_db]) * 10
    )

    print("Solving Global Schedule (this may take a moment)...")
    prob.solve(pulp.PULP_CBC_CMD(msg=1, timeLimit=60))
    
    # --- 7. RESULTS ---
    res = []
    for (a, d, s), var in choices.items():
        if pulp.value(var) == 1:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            res.append({
                "Date": pd.to_datetime(d).date(),
                "Agent": a,
                "Start": f"{s:02d}:00",
                "End": f"{(s+dur)%24:02d}:00"
            })
    return res

# --- MAIN EXECUTION ---
try:
    df = pd.read_excel("need.xlsx")
    clean_df = pd.DataFrame()
    clean_df['Date'] = pd.to_datetime(df.iloc[:, 0])
    clean_df['Hour'] = pd.to_numeric(df.iloc[:, 2].astype(str).str.split(':').str[0], errors='coerce')
    clean_df['FTE_Brut'] = pd.to_numeric(df.iloc[:, 4], errors='coerce').fillna(0)
    clean_df = clean_df.dropna(subset=['Hour'])

    final_results = solve_full_period(clean_df)

    if final_results:
        output_df = pd.DataFrame(final_results)
        output_df.to_excel("final_schedule_global.xlsx", index=False)
        print(f"\n*** SUCCESS! {len(final_results)} assignments made. ***")
    else:
        print("\n*** ERROR: No assignments generated. ***")

except Exception:
    import traceback
    print(traceback.format_exc())

Solving Global Schedule (this may take a moment)...

*** SUCCESS! 1346 assignments made. ***


In [4]:
import pulp
import pandas as pd
from datetime import datetime

# --- 1. CONFIGURATION ---
agents_db = {}
for i in range(1, 51):
    if i <= 20: 
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (0, 9)}
    elif i <= 35: 
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (9, 23)}
    else: 
        agents_db[f"Agent_{i:02d}"] = {"type": "PT", "workdays": [0,1,6], "window": (0, 23)}

def solve_full_period(clean_df):
    all_dates = sorted(clean_df['Date'].unique())
    prob = pulp.LpProblem("Global_Schedule", pulp.LpMinimize)
    
    choices = {}      # (agent, date, start_hour)
    slack_vars = {}   # (date, hour)
    agent_daily_work = {} # (agent, date) -> Binary (1 if working that day)

    # --- 2. VARIABLE CREATION ---
    for d in all_dates:
        ts = pd.to_datetime(d)
        weekday = ts.weekday()
        avail_today = [a for a, info in agents_db.items() if weekday in info['workdays']]
        
        for a in avail_today:
            s_min, s_max = agents_db[a]['window']
            # Track if agent works at all on this day
            agent_daily_work[(a, d)] = pulp.LpVariable(f"Work_{a}_{d}", cat="Binary")
            
            day_choices = []
            for s in range(s_min, s_max + 1):
                v = pulp.LpVariable(f"S_{a}_{d}_{s}", cat="Binary")
                choices[(a, d, s)] = v
                day_choices.append(v)
            
            # Constraint: Max 1 shift per day
            prob += pulp.lpSum(day_choices) == agent_daily_work[(a, d)]

    # --- 3. COVERAGE & SLACK ---
    for d in all_dates:
        day_data = clean_df[clean_df['Date'] == d]
        demand_dict = dict(zip(day_data['Hour'], day_data['FTE_Brut']))
        
        for h in range(24):
            coverage = []
            for (a, date, s), var in choices.items():
                if date == d:
                    dur = 8 if agents_db[a]['type'] == "FT" else 4
                    if s <= h < s + dur:
                        # Handle Lunch for FT
                        if agents_db[a]['type'] == "FT" and h == (s + 4):
                            continue
                        coverage.append(var)
            
            slack = pulp.LpVariable(f"Slack_{d}_{h}", lowBound=0)
            slack_vars[(d, h)] = slack
            prob += pulp.lpSum(coverage) + slack >= demand_dict.get(h, 0)

    # --- 4. LAYER: EQUITY (Rebalancing) ---
    # Target: Total shifts needed / Number of agents
    total_days = len(all_dates)
    for a in agents_db:
        # Number of shifts assigned to this agent over the period
        agent_total_shifts = pulp.lpSum([agent_daily_work[(a, d)] for d in all_dates if (a, d) in agent_daily_work])
        
        # Helper variables to measure deviation from a "fair" load
        # We penalize agents working way more or way less than 4 shifts a week (as an example)
        over_work = pulp.LpVariable(f"Over_{a}", lowBound=0)
        prob += agent_total_shifts <= 5 + over_work # Penalty if more than 5 shifts/week

    # --- 5. LAYER: NO 3-DAY GAPS ---
    # For every agent, in any rolling 3-day window they are available, they must work >= 1 day
    for a in agents_db:
        workdays = agents_db[a]['workdays']
        for i in range(len(all_dates) - 2):
            window = all_dates[i:i+3]
            # Only apply if the agent is actually contracted to work these days
            valid_days_in_window = [d for d in window if pd.to_datetime(d).weekday() in workdays]
            
            if len(valid_days_in_window) == 3:
                prob += pulp.lpSum([agent_daily_work[(a, d)] for d in valid_days_in_window]) >= 1

    # --- 6. OBJECTIVE ---
    # 1. Minimize Understaffing (Slack) - Priority 1
    # 2. Minimize Total Agents used - Priority 2
    # 3. Minimize Overwork - Priority 3
    prob += (
        pulp.lpSum(slack_vars.values()) * 100 + 
        pulp.lpSum(choices.values()) * 1 +
        pulp.lpSum([pulp.LpVariable(f"dev_{a}") for a in agents_db]) * 10
    )

    print("Solving Global Schedule (this may take a moment)...")
    prob.solve(pulp.PULP_CBC_CMD(msg=1, timeLimit=60))
    
    # --- 7. RESULTS ---
    res = []
    for (a, d, s), var in choices.items():
        if pulp.value(var) == 1:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            res.append({
                "Date": pd.to_datetime(d).date(),
                "Agent": a,
                "Start": f"{s:02d}:00",
                "End": f"{(s+dur)%24:02d}:00"
            })
    return res

# --- MAIN EXECUTION ---
try:
    df = pd.read_excel("need.xlsx")
    clean_df = pd.DataFrame()
    clean_df['Date'] = pd.to_datetime(df.iloc[:, 0])
    clean_df['Hour'] = pd.to_numeric(df.iloc[:, 2].astype(str).str.split(':').str[0], errors='coerce')
    clean_df['FTE_Brut'] = pd.to_numeric(df.iloc[:, 4], errors='coerce').fillna(0)
    clean_df = clean_df.dropna(subset=['Hour'])

    final_results = solve_full_period(clean_df)

    if final_results:
        output_df = pd.DataFrame(final_results)
        output_df.to_excel("final_schedule_global.xlsx", index=False)
        print(f"\n*** SUCCESS! {len(final_results)} assignments made. ***")
    else:
        print("\n*** ERROR: No assignments generated. ***")

except Exception:
    import traceback
    print(traceback.format_exc())

Solving Global Schedule (this may take a moment)...
Traceback (most recent call last):
  File "C:\Users\B622UD\AppData\Local\Temp\ipykernel_19732\4183339717.py", line 126, in <module>
    output_df.to_excel("final_schedule_global.xlsx", index=False)
  File "C:\Users\B622UD\AppData\Local\anaconda3\Lib\site-packages\pandas\core\generic.py", line 2252, in to_excel
    formatter.write(
  File "C:\Users\B622UD\AppData\Local\anaconda3\Lib\site-packages\pandas\io\formats\excel.py", line 934, in write
    writer = ExcelWriter(  # type: ignore[abstract]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\B622UD\AppData\Local\anaconda3\Lib\site-packages\pandas\io\excel\_xlsxwriter.py", line 199, in __init__
    super().__init__(
  File "C:\Users\B622UD\AppData\Local\anaconda3\Lib\site-packages\pandas\io\excel\_base.py", line 1219, in __init__
    self._handles = get_handle(
                    ^^^^^^^^^^^
  File "C:\Users\B622UD\AppData\Local\anaconda3\Lib\site-packages\pandas\i

In [5]:
import pulp
import pandas as pd
from datetime import datetime

# --- 1. CONFIGURATION ---
agents_db = {}
for i in range(1, 51):
    if i <= 20: 
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (0, 9)}
    elif i <= 35: 
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (9, 23)}
    else: 
        agents_db[f"Agent_{i:02d}"] = {"type": "PT", "workdays": [0,1,6], "window": (0, 23)}

def solve_full_period(clean_df):
    all_dates = sorted(clean_df['Date'].unique())
    prob = pulp.LpProblem("Global_Schedule", pulp.LpMinimize)
    
    choices = {}      # (agent, date, start_hour)
    slack_vars = {}   # (date, hour)
    agent_daily_work = {} # (agent, date) -> Binary (1 if working that day)

    # --- 2. VARIABLE CREATION ---
    for d in all_dates:
        ts = pd.to_datetime(d)
        weekday = ts.weekday()
        avail_today = [a for a, info in agents_db.items() if weekday in info['workdays']]
        
        for a in avail_today:
            s_min, s_max = agents_db[a]['window']
            # Track if agent works at all on this day
            agent_daily_work[(a, d)] = pulp.LpVariable(f"Work_{a}_{d}", cat="Binary")
            
            day_choices = []
            for s in range(s_min, s_max + 1):
                v = pulp.LpVariable(f"S_{a}_{d}_{s}", cat="Binary")
                choices[(a, d, s)] = v
                day_choices.append(v)
            
            # Constraint: Max 1 shift per day
            prob += pulp.lpSum(day_choices) == agent_daily_work[(a, d)]

    # --- 3. COVERAGE & SLACK ---
    for d in all_dates:
        day_data = clean_df[clean_df['Date'] == d]
        demand_dict = dict(zip(day_data['Hour'], day_data['FTE_Brut']))
        
        for h in range(24):
            coverage = []
            for (a, date, s), var in choices.items():
                if date == d:
                    dur = 8 if agents_db[a]['type'] == "FT" else 4
                    if s <= h < s + dur:
                        # Handle Lunch for FT
                        if agents_db[a]['type'] == "FT" and h == (s + 4):
                            continue
                        coverage.append(var)
            
            slack = pulp.LpVariable(f"Slack_{d}_{h}", lowBound=0)
            slack_vars[(d, h)] = slack
            prob += pulp.lpSum(coverage) + slack >= demand_dict.get(h, 0)

    # --- 4. LAYER: EQUITY (Rebalancing) ---
    # Target: Total shifts needed / Number of agents
    total_days = len(all_dates)
    for a in agents_db:
        # Number of shifts assigned to this agent over the period
        agent_total_shifts = pulp.lpSum([agent_daily_work[(a, d)] for d in all_dates if (a, d) in agent_daily_work])
        
        # Helper variables to measure deviation from a "fair" load
        # We penalize agents working way more or way less than 4 shifts a week (as an example)
        over_work = pulp.LpVariable(f"Over_{a}", lowBound=0)
        prob += agent_total_shifts <= 5 + over_work # Penalty if more than 5 shifts/week

    # --- 5. LAYER: NO 3-DAY GAPS ---
    # For every agent, in any rolling 3-day window they are available, they must work >= 1 day
    for a in agents_db:
        workdays = agents_db[a]['workdays']
        for i in range(len(all_dates) - 2):
            window = all_dates[i:i+3]
            # Only apply if the agent is actually contracted to work these days
            valid_days_in_window = [d for d in window if pd.to_datetime(d).weekday() in workdays]
            
            if len(valid_days_in_window) == 3:
                prob += pulp.lpSum([agent_daily_work[(a, d)] for d in valid_days_in_window]) >= 1

    # --- 6. OBJECTIVE ---
    # 1. Minimize Understaffing (Slack) - Priority 1
    # 2. Minimize Total Agents used - Priority 2
    # 3. Minimize Overwork - Priority 3
    prob += (
        pulp.lpSum(slack_vars.values()) * 100 + 
        pulp.lpSum(choices.values()) * 1 +
        pulp.lpSum([pulp.LpVariable(f"dev_{a}") for a in agents_db]) * 10
    )

    print("Solving Global Schedule (this may take a moment)...")
    prob.solve(pulp.PULP_CBC_CMD(msg=1, timeLimit=120))
    
    # --- 7. RESULTS ---
    res = []
    for (a, d, s), var in choices.items():
        if pulp.value(var) == 1:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            res.append({
                "Date": pd.to_datetime(d).date(),
                "Agent": a,
                "Start": f"{s:02d}:00",
                "End": f"{(s+dur)%24:02d}:00"
            })
    return res

# --- MAIN EXECUTION ---
try:
    df = pd.read_excel("need.xlsx")
    clean_df = pd.DataFrame()
    clean_df['Date'] = pd.to_datetime(df.iloc[:, 0])
    clean_df['Hour'] = pd.to_numeric(df.iloc[:, 2].astype(str).str.split(':').str[0], errors='coerce')
    clean_df['FTE_Brut'] = pd.to_numeric(df.iloc[:, 4], errors='coerce').fillna(0)
    clean_df = clean_df.dropna(subset=['Hour'])

    final_results = solve_full_period(clean_df)

    if final_results:
        output_df = pd.DataFrame(final_results)
        output_df.to_excel("final_schedule_global.xlsx", index=False)
        print(f"\n*** SUCCESS! {len(final_results)} assignments made. ***")
    else:
        print("\n*** ERROR: No assignments generated. ***")

except Exception:
    import traceback
    print(traceback.format_exc())

Solving Global Schedule (this may take a moment)...

*** SUCCESS! 1242 assignments made. ***


In [1]:
import pulp
import pandas as pd
from datetime import datetime

# --- 1. CONFIGURATION ---
agents_db = {}
for i in range(1, 51):
    if i <= 25: 
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (8, 16)}
    elif i <= 15: 
        agents_db[f"Agent_{i:02d}"] = {"type": "FT", "workdays": [0,1,2,3,4], "window": (16, 0)}
    else: 
        agents_db[f"Agent_{i:02d}"] = {"type": "PT", "workdays": [0,1,2,3,4], "window": (0, 8)}

def solve_full_period(clean_df):
    all_dates = sorted(clean_df['Date'].unique())
    prob = pulp.LpProblem("Global_Schedule", pulp.LpMinimize)
    
    choices = {}      # (agent, date, start_hour)
    slack_vars = {}   # (date, hour)
    agent_daily_work = {} # (agent, date) -> Binary (1 if working that day)

    # --- 2. VARIABLE CREATION ---
    for d in all_dates:
        ts = pd.to_datetime(d)
        weekday = ts.weekday()
        avail_today = [a for a, info in agents_db.items() if weekday in info['workdays']]
        
        for a in avail_today:
            s_min, s_max = agents_db[a]['window']
            # Track if agent works at all on this day
            agent_daily_work[(a, d)] = pulp.LpVariable(f"Work_{a}_{d}", cat="Binary")
            
            day_choices = []
            for s in range(s_min, s_max + 1):
                v = pulp.LpVariable(f"S_{a}_{d}_{s}", cat="Binary")
                choices[(a, d, s)] = v
                day_choices.append(v)
            
            # Constraint: Max 1 shift per day
            prob += pulp.lpSum(day_choices) == agent_daily_work[(a, d)]

    # --- 3. COVERAGE & SLACK ---
    for d in all_dates:
        day_data = clean_df[clean_df['Date'] == d]
        demand_dict = dict(zip(day_data['Hour'], day_data['FTE_Brut']))
        
        for h in range(24):
            coverage = []
            for (a, date, s), var in choices.items():
                if date == d:
                    dur = 8 if agents_db[a]['type'] == "FT" else 4
                    if s <= h < s + dur:
                        # Handle Lunch for FT
                        if agents_db[a]['type'] == "FT" and h == (s + 4):
                            continue
                        coverage.append(var)
            
            slack = pulp.LpVariable(f"Slack_{d}_{h}", lowBound=0)
            slack_vars[(d, h)] = slack
            prob += pulp.lpSum(coverage) + slack >= demand_dict.get(h, 0)

    # --- 4. LAYER: EQUITY (Rebalancing) ---
    # Target: Total shifts needed / Number of agents
    total_days = len(all_dates)
    for a in agents_db:
        # Number of shifts assigned to this agent over the period
        agent_total_shifts = pulp.lpSum([agent_daily_work[(a, d)] for d in all_dates if (a, d) in agent_daily_work])
        
        # Helper variables to measure deviation from a "fair" load
        # We penalize agents working way more or way less than 4 shifts a week (as an example)
        over_work = pulp.LpVariable(f"Over_{a}", lowBound=0)
        prob += agent_total_shifts <= 5 + over_work # Penalty if more than 5 shifts/week

    # --- 5. LAYER: NO 3-DAY GAPS ---
    # For every agent, in any rolling 3-day window they are available, they must work >= 1 day
    for a in agents_db:
        workdays = agents_db[a]['workdays']
        for i in range(len(all_dates) - 2):
            window = all_dates[i:i+3]
            # Only apply if the agent is actually contracted to work these days
            valid_days_in_window = [d for d in window if pd.to_datetime(d).weekday() in workdays]
            
            if len(valid_days_in_window) == 3:
                prob += pulp.lpSum([agent_daily_work[(a, d)] for d in valid_days_in_window]) >= 1

    # --- 6. OBJECTIVE ---
    # 1. Minimize Understaffing (Slack) - Priority 1
    # 2. Minimize Total Agents used - Priority 2
    # 3. Minimize Overwork - Priority 3
    prob += (
        pulp.lpSum(slack_vars.values()) * 100 + 
        pulp.lpSum(choices.values()) * 1 +
        pulp.lpSum([pulp.LpVariable(f"dev_{a}") for a in agents_db]) * 10
    )

    print("Solving Global Schedule (this may take a moment)...")
    prob.solve(pulp.PULP_CBC_CMD(msg=1, timeLimit=240))
    
    # --- 7. RESULTS ---
    res = []
    for (a, d, s), var in choices.items():
        if pulp.value(var) == 1:
            dur = 8 if agents_db[a]['type'] == "FT" else 4
            res.append({
                "Date": pd.to_datetime(d).date(),
                "Agent": a,
                "Start": f"{s:02d}:00",
                "End": f"{(s+dur)%24:02d}:00"
            })
    return res

# --- MAIN EXECUTION ---
try:
    df = pd.read_excel("need.xlsx")
    clean_df = pd.DataFrame()
    clean_df['Date'] = pd.to_datetime(df.iloc[:, 0])
    clean_df['Hour'] = pd.to_numeric(df.iloc[:, 2].astype(str).str.split(':').str[0], errors='coerce')
    clean_df['FTE_Brut'] = pd.to_numeric(df.iloc[:, 4], errors='coerce').fillna(0)
    clean_df = clean_df.dropna(subset=['Hour'])

    final_results = solve_full_period(clean_df)

    if final_results:
        output_df = pd.DataFrame(final_results)
        output_df.to_excel("final_schedule_global.xlsx", index=False)
        print(f"\n*** SUCCESS! {len(final_results)} assignments made. ***")
    else:
        print("\n*** ERROR: No assignments generated. ***")

except Exception:
    import traceback
    print(traceback.format_exc())

Solving Global Schedule (this may take a moment)...

*** SUCCESS! 1915 assignments made. ***
